# Nifty 50 — Variance, Standard Deviation And What India VIX Was Charging For It

Realised variance and standard deviation for Nifty 50, held against **India VIX** — the market's *implied* volatility for the same risk.

The two series are different kinds of number and the whole notebook turns on keeping them straight:

- **Realised volatility (RV)** is backward looking. It is the standard deviation of returns that actually happened, annualised by √252 so it is quoted in the same units as VIX.
- **India VIX** is forward looking. NSE computes it from Nifty option prices as the annualised standard deviation the option market is pricing for the **next 30 calendar days** (≈ 21 trading sessions). A VIX of 15 is a claim that the index will move with a 15% annualised σ over the next month.
- The gap between the two is the **variance risk premium** — what an option seller is paid for carrying the risk, and the reason this repo's short-premium notebooks work at all.

Because VIX is a forecast, the honest comparison is VIX today against realised vol over the **following** `RV_WINDOW` sessions, not the trailing window. Both are computed here; the trailing one is for the time-series chart, the forward one for every premium number.

What this notebook answers:

1. Daily and annualised **variance and standard deviation** for Nifty 50, from close-to-close returns and from four other estimators that disagree in an informative way.
2. How much of that variance arrives **overnight**, when no stop can act.
3. The **variance risk premium** — in vol points and in variance points — overall and conditioned on the VIX regime.
4. Whether the VIX-implied **one-session move** is a fair description of the next day's actual move, against both the realised record and a Gaussian benchmark.
5. How much skill VIX actually has as a forecast, and how VIX behaves as a series in its own right.

India VIX is computed from Nifty 50 option prices, so for this index the implied and the realised series refer to the same underlying and no proxy assumption is involved. The Sensex companion notebook carries one extra section, because there that assumption does need testing.

Source data (hourly OHLC, both starting 2022-01-03 despite the file names):

- `data/raw/option_selling/nifty/nifty50_hourly_2020-01-01_2026-11-06.json`
- `data/raw/option_selling/india_vix_hourly_2020-01-01_2026-11-06.json`

Every figure below is computed from these files; nothing is hard-coded in the prose. The companion notebook for Sensex is `notebooks/quant/sensex/variance_stddev_vs_india_vix.ipynb` and runs the identical code path.

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the notebook's location until the folder holding data/raw is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "option_selling"

# --- Which index this notebook measures ------------------------------------
INDEX_NAME = "NIFTY 50"
INDEX_JSON = RAW_DIR / "nifty" / "nifty50_hourly_2020-01-01_2026-11-06.json"
INDEX_HEX = "#2a78d6"          # this index's fixed categorical hue
# ---------------------------------------------------------------------------

VIX_JSON = RAW_DIR / "india_vix_hourly_2020-01-01_2026-11-06.json"
VIX_NAME = "INDIA VIX"

# --- Parameters -------------------------------------------------------------
TRADING_DAYS = 252                                 # sessions per year, used to annualise every sigma
CANDLES_PER_SESSION = 7                            # hourly candles in a regular session (09:15 .. 15:15)
RV_WINDOW = 21                                     # sessions of realised vol set against VIX (~30 calendar days)
VIX_BANDS = [0, 12, 14, 16, 18, 22, 100]           # VIX regime edges, in annualised vol points
SIGMA_MULTIPLES = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]   # tested against the VIX-implied one-session sigma
REGULAR_SESSION_END_HOUR = 16                      # candles stamped later are Muhurat/special sessions
REGULAR_SESSION_MIN_CANDLES = 6                    # weekday sessions with fewer candles are not normal trading days
# ---------------------------------------------------------------------------

ANNUALISE = math.sqrt(TRADING_DAYS)

VIX_HEX = "#1baf7a"                                # India VIX, held in one hue across both notebooks
# Diverging poles for signed quantities: VIX richer than realised = warm, cheaper = cool.
RICH_COLOR, CHEAP_COLOR, NEUTRAL = (235, 104, 52), (42, 120, 214), (240, 239, 236)
RICH_HEX, CHEAP_HEX = "#eb6834", "#2a78d6"
SEQUENTIAL = [[0, "#cde2fb"], [0.5, "#3987e5"], [1, "#0d366b"]]   # one hue, light to dark
TERCILE_HEX = ["#86b6ef", "#256abf", "#104281"]                   # ordered scale: one hue, light to dark
GRID, MUTED_INK, HEADER_FILL = "#eceae6", "#8a8880", "#2b2d42"

pd.options.display.float_format = "{:,.2f}".format

## Load The Index And VIX Onto One Session Calendar

Both dumps share the same hourly grid, so the clean-up is the same for each:

- **Special sessions are dropped.** Three kinds turn up in these dumps: Muhurat prints (a single candle stamped at 18:00/18:15, or one stray afternoon candle), partial weekend drills (three candles), and *full* weekend special sessions such as the Budget Saturdays, which carry a complete seven-candle grid and so cannot be caught by a candle count. A regular session is therefore a **weekday** with at least `REGULAR_SESSION_MIN_CANDLES` candles stamped before `REGULAR_SESSION_END_HOUR` — the weekday test removes the weekend sessions, the candle count the Muhurat stubs. Left in, a one-candle session would register as an enormous or a zero return. The cell prints exactly which dates were removed.
- **Hourly candles are collapsed into one OHLC row per session** — first open, highest high, lowest low, last close. Taking the daily high as the max of the hourly highs is exact, so the daily range estimators below are not degraded by the hourly source.
- **The calendar is intersected across both series.** Every comparison then runs on sessions where an index close *and* a VIX close both exist, so no statistic is computed over a different sample than the one it is compared against.

VIX is loaded through the same OHLC path, but note it is a *level*, not a price: its close is the quoted annualised volatility, and only that close is used.

In [2]:
def load_hourly(path: Path) -> pd.DataFrame:
    """Hourly OHLC from a candle dump, de-duplicated and stamped in IST-naive time."""
    with open(path) as f:
        raw = json.load(f)
    frame = pd.DataFrame(
        raw["data"]["candles"],
        columns=["timestamp", "open", "high", "low", "close", "volume", "oi"],
    )
    frame["timestamp"] = (
        pd.to_datetime(frame["timestamp"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
    )
    frame = frame.drop_duplicates("timestamp").set_index("timestamp").sort_index()
    return frame[["open", "high", "low", "close"]].apply(pd.to_numeric)


def regular_sessions(hourly_frame: pd.DataFrame) -> pd.DatetimeIndex:
    """Weekday sessions carrying a full hourly grid — Muhurat and special sessions excluded."""
    intraday = hourly_frame[hourly_frame.index.hour < REGULAR_SESSION_END_HOUR]
    per_session = intraday.groupby(intraday.index.normalize()).size()
    return per_session.index[
        (per_session.index.dayofweek < 5) & (per_session.to_numpy() >= REGULAR_SESSION_MIN_CANDLES)
    ]


def session_candles(hourly_frame: pd.DataFrame, keep: pd.DatetimeIndex) -> pd.DataFrame:
    """Hourly candles of the kept sessions, tagged with the session they belong to."""
    intraday = hourly_frame[hourly_frame.index.hour < REGULAR_SESSION_END_HOUR].copy()
    intraday["session"] = intraday.index.normalize()
    return intraday[intraday["session"].isin(keep)]


def to_daily(hourly_frame: pd.DataFrame, keep: pd.DatetimeIndex) -> pd.DataFrame:
    """Collapse hourly candles into one OHLC row per regular session."""
    grouped = session_candles(hourly_frame, keep).groupby("session")
    return pd.DataFrame(
        {
            "open": grouped["open"].first(),
            "high": grouped["high"].max(),
            "low": grouped["low"].min(),
            "close": grouped["close"].last(),
        }
    )


raw_hourly = {INDEX_NAME: load_hourly(INDEX_JSON), VIX_NAME: load_hourly(VIX_JSON)}

sessions = regular_sessions(raw_hourly[INDEX_NAME])
for frame in raw_hourly.values():
    sessions = sessions.intersection(regular_sessions(frame))

hourly = {name: session_candles(frame, sessions) for name, frame in raw_hourly.items()}
daily = {name: to_daily(frame, sessions) for name, frame in raw_hourly.items()}
price = daily[INDEX_NAME]
vix = daily[VIX_NAME]["close"].rename("vix")

for name, frame in raw_hourly.items():
    dropped = sorted(set(frame.index.normalize().unique()) - set(sessions))
    print(
        f"{name:<10} {len(frame):>6,} hourly candles -> {len(daily[name]):>5,} regular sessions, "
        f"{len(dropped)} session(s) dropped: " + ", ".join(f"{d:%Y-%m-%d %a}" for d in dropped)
    )

print(
    f"\nCommon calendar: {len(sessions):,} sessions, {sessions.min():%Y-%m-%d} to {sessions.max():%Y-%m-%d} "
    f"({(sessions.max() - sessions.min()).days / 365.25:.2f} years)"
)
print(f"Candles per kept session: {hourly[INDEX_NAME].groupby('session').size().unique()}")
print(f"{INDEX_NAME} close: {price['close'].min():,.2f} to {price['close'].max():,.2f}")
print(f"India VIX close: min {vix.min():.2f}, median {vix.median():.2f}, mean {vix.mean():.2f}, max {vix.max():.2f}")

NIFTY 50    8,305 hourly candles -> 1,182 regular sessions, 9 session(s) dropped: 2022-10-24 Mon, 2023-11-12 Sun, 2024-01-20 Sat, 2024-03-02 Sat, 2024-05-18 Sat, 2024-11-01 Fri, 2025-02-01 Sat, 2025-10-21 Tue, 2026-02-01 Sun
INDIA VIX   8,305 hourly candles -> 1,182 regular sessions, 9 session(s) dropped: 2022-10-24 Mon, 2023-11-12 Sun, 2024-01-20 Sat, 2024-03-02 Sat, 2024-05-18 Sat, 2024-11-01 Fri, 2025-02-01 Sat, 2025-10-21 Tue, 2026-02-01 Sun

Common calendar: 1,182 sessions, 2022-01-03 to 2026-10-22 (4.80 years)
Candles per kept session: [7]
NIFTY 50 close: 15,330.75 to 26,335.70
India VIX close: min 9.13, median 13.90, mean 14.94, max 32.75


## Return Construction

Everything downstream is built from **log returns**, `ln(Pₜ / Pₜ₋₁)`. Log returns are the right unit here because they add across time, which is what lets a one-session variance be scaled to a year by √252 and compared against a VIX quote.

Four return series, each answering a different question:

| Series | Definition | What it measures |
|---|---|---|
| `close_to_close` | `ln(Cₜ / Cₜ₋₁)` | the full session-over-session move — the benchmark σ, and the only one directly comparable to VIX |
| `overnight` | `ln(Oₜ / Cₜ₋₁)` | the gap, which no intraday stop can act on |
| `intraday` | `ln(Cₜ / Oₜ)` | open-to-close, the part a position can be managed through |
| `log_range` | `ln(Hₜ / Lₜ)` | the session's travel, input to the range estimators |

The split is exact — `overnight + intraday == close_to_close` by construction — which the assertion below enforces. That matters because the variance of a sum is *not* the sum of variances: the cross term is reported explicitly rather than quietly dropped.

Hourly steps are built separately, and one detail of the grid matters. A session has seven candles stamped 09:15 through 15:15, but the cash session ends at 15:30 — so the first six cover a full hour each and the **last covers only fifteen minutes**. The steps are therefore not equal-length intervals. Taking the first step of each session as that candle's own open-to-close makes the seven steps sum exactly to the session's intraday return, with no overnight gap leaking in, but the short final candle should be kept in mind wherever per-candle figures are compared.

In [3]:
def return_set(frame: pd.DataFrame) -> pd.DataFrame:
    """Log returns per session: the close-to-close move and its exact overnight/intraday split."""
    built = pd.DataFrame(
        {
            "close_to_close": np.log(frame["close"]).diff(),
            "overnight": np.log(frame["open"] / frame["close"].shift()),
            "intraday": np.log(frame["close"] / frame["open"]),
            "log_range": np.log(frame["high"] / frame["low"]),
            "log_close_open": np.log(frame["close"] / frame["open"]),
        }
    )
    split = built["overnight"] + built["intraday"]
    assert np.allclose(split.dropna(), built["close_to_close"].dropna()), "overnight + intraday must rebuild the daily return"
    return built


def hourly_log_returns(name: str) -> pd.Series:
    """Hourly log returns inside each session.

    The first step of a session is that candle's own open-to-close, later steps are
    close-to-close, so the steps of one session sum exactly to its intraday log return
    and no overnight gap leaks into the hourly sample.
    """
    candles = hourly[name]
    first_of_session = ~candles["session"].eq(candles["session"].shift())
    base = candles["close"].shift().where(~first_of_session, candles["open"])
    return np.log(candles["close"] / base)


returns = return_set(price)
daily_returns = returns["close_to_close"].dropna()
overnight, intraday = returns["overnight"].dropna(), returns["intraday"].dropna()
steps = hourly_log_returns(INDEX_NAME).dropna()

print(
    f"{INDEX_NAME}: {len(daily_returns):,} daily returns, {len(steps):,} hourly steps "
    f"({len(steps) / len(price):.1f} per session)"
)
print(f"Overnight + intraday rebuilds the daily return exactly: assertion passed")

NIFTY 50: 1,181 daily returns, 8,274 hourly steps (7.0 per session)
Overnight + intraday rebuilds the daily return exactly: assertion passed


## Headline Variance And Standard Deviation

Five estimators of the same quantity, and they do not agree — the disagreement is the result, not noise:

- **Close-to-close σ** is the benchmark. It is the standard deviation of `ln(Cₜ/Cₜ₋₁)`, reported per session and annualised by √252. This is the number VIX is quoting against.
- **Overnight σ** and **intraday σ** split it. Their variances plus the cross term `2·Cov(overnight, intraday)` rebuild the daily variance exactly, so all three shares are shown and sum to 100%. The shares can individually exceed 100% when the cross term is negative — that is the session partly retracing its own gap, and it is reported rather than hidden.
- **Parkinson σ** uses only the high-low range, `σ² = E[ln(H/L)²] / (4·ln2)`. The range is a far more efficient estimator than a single close — but it sees only what happened *between* the open and the close.
- **Garman–Klass σ** adds the close-to-open term, `σ² = E[0.5·ln(H/L)² − (2ln2−1)·ln(C/O)²]`, and is more efficient again. It is also **intraday-only**.

That last point explains the gap you are about to see: Parkinson and Garman–Klass both come in well *below* close-to-close σ, and they are not wrong. They measure the session's travel from open to close and are structurally blind to the overnight gap. Close-to-close σ includes it. The difference is the gap risk, and the variance decomposition sizes it directly.

**Variance is reported alongside σ throughout.** Variance is the additive quantity — it is what scales linearly with time, what a variance swap settles on, and what the `VIX²` comparisons later in the notebook use. σ is the readable quantity. Daily variance is in %², annualised variance is daily variance × 252.

In [4]:
def parkinson_sigma(frame: pd.DataFrame) -> float:
    """Annualised % sigma from the high-low range alone (Parkinson 1980). Intraday only."""
    log_range = frame["log_range"].dropna()
    return math.sqrt((log_range**2).mean() / (4 * math.log(2))) * ANNUALISE * 100


def garman_klass_sigma(frame: pd.DataFrame) -> float:
    """Annualised % sigma from range plus close-to-open (Garman-Klass 1980). Intraday only."""
    both = frame[["log_range", "log_close_open"]].dropna()
    variance = (
        0.5 * both["log_range"] ** 2 - (2 * math.log(2) - 1) * both["log_close_open"] ** 2
    ).mean()
    return math.sqrt(variance) * ANNUALISE * 100


def trailing_realised_vol(series: pd.Series, window: int = RV_WINDOW) -> pd.Series:
    """Annualised % sigma of the `window` returns up to and including each session."""
    return series.rolling(window).std() * ANNUALISE * 100


def forward_realised_vol(series: pd.Series, window: int = RV_WINDOW) -> pd.Series:
    """Annualised % sigma of the `window` returns falling strictly AFTER each session.

    This is the quantity VIX is a forecast of, so it is what every premium number below
    is measured against. Sessions without a full forward window are left as NaN.
    """
    clean = series.dropna()
    values = clean.to_numpy(dtype=float)
    out = np.full(len(values), np.nan)
    if len(values) > window:
        future = np.lib.stride_tricks.sliding_window_view(values[1:], window)
        out[: len(future)] = future.std(axis=1, ddof=1)
    return pd.Series(out * ANNUALISE * 100, index=clean.index).reindex(series.index)


trailing_rv = trailing_realised_vol(returns["close_to_close"])
forward_rv = forward_realised_vol(returns["close_to_close"])

daily_sigma_pct = daily_returns.std() * 100
daily_variance = daily_sigma_pct**2
daily_var_raw = daily_returns.var()
cross_term = 2 * overnight.cov(intraday)

rows = [
    ("Sessions in sample", f"{len(price):,}", "weekdays with a full hourly grid"),
    ("Daily returns", f"{len(daily_returns):,}", "one fewer than sessions"),
    ("Mean daily log return", f"{daily_returns.mean() * 100:+.4f}%", "drift, immaterial at this scale"),
    ("__Close-to-close", "", ""),
    ("Daily sigma", f"{daily_sigma_pct:.4f}%", "the benchmark σ, per session"),
    ("Daily variance", f"{daily_variance:.4f} %²", "σ² — the additive quantity"),
    ("Annualised sigma", f"{daily_sigma_pct * ANNUALISE:.2f}%", "×√252, directly comparable to VIX"),
    ("Annualised variance", f"{daily_variance * TRADING_DAYS:,.1f} %²", "daily variance × 252"),
    ("Mean absolute daily move", f"{daily_returns.abs().mean() * 100:.3f}%", "what a seller actually faces"),
    ("Median absolute daily move", f"{daily_returns.abs().median() * 100:.3f}%", "below the mean — right-skewed"),
    ("Skew", f"{daily_returns.skew():+.2f}", "negative: down moves are larger"),
    ("Excess kurtosis", f"{daily_returns.kurtosis():+.2f}", "fat tails vs a normal"),
    ("Worst session", f"{daily_returns.min() * 100:+.2f}%", ""),
    ("Best session", f"{daily_returns.max() * 100:+.2f}%", ""),
    ("__Where the variance comes from", "", ""),
    ("Overnight sigma (annualised)", f"{overnight.std() * ANNUALISE * 100:.2f}%", "the gap, unmanageable"),
    ("Intraday sigma (annualised)", f"{intraday.std() * ANNUALISE * 100:.2f}%", "open-to-close, manageable"),
    ("Overnight share of daily variance", f"{overnight.var() / daily_var_raw * 100:.1f}%", ""),
    ("Intraday share of daily variance", f"{intraday.var() / daily_var_raw * 100:.1f}%", ""),
    ("Cross-term share (2·Cov)", f"{cross_term / daily_var_raw * 100:+.1f}%", "negative = the gap partly retraces"),
    ("__Other estimators (annualised)", "", ""),
    ("Parkinson sigma (range only)", f"{parkinson_sigma(returns):.2f}%", "intraday only — blind to the gap"),
    ("Garman–Klass sigma", f"{garman_klass_sigma(returns):.2f}%", "intraday only — blind to the gap"),
    (
        "Hourly-sampled intraday sigma",
        f"{steps.std() * math.sqrt(TRADING_DAYS * CANDLES_PER_SESSION) * 100:.2f}%",
        "close to intraday σ ⇒ near-iid hours",
    ),
    ("__Against India VIX", "", ""),
    ("Mean India VIX", f"{vix.mean():.2f}", "implied, forward 30 calendar days"),
    (f"Mean forward {RV_WINDOW}-session realised sigma", f"{forward_rv.mean():.2f}%", "what VIX was forecasting"),
    ("Mean variance risk premium", f"{vix.mean() - forward_rv.mean():+.2f} vol pts", "implied minus realised"),
]

labels = [r[0] for r in rows]
is_group = [label.startswith("__") for label in labels]

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[250, 120, 260],
            header=dict(
                values=["Metric", INDEX_NAME, "Note"],
                fill_color=HEADER_FILL,
                font=dict(color="white", size=12),
                align=["left", "right", "left"],
                height=32,
            ),
            cells=dict(
                values=[
                    [label.removeprefix("__") for label in labels],
                    [r[1] for r in rows],
                    [r[2] for r in rows],
                ],
                fill_color=[["#e7e9f2" if g else "white" for g in is_group]] * 3,
                font=dict(
                    color=[
                        ["#2b2d42" if g else "#333231" for g in is_group],
                        ["#2b2d42" if g else "#333231" for g in is_group],
                        ["#2b2d42" if g else MUTED_INK for g in is_group],
                    ],
                    size=11,
                ),
                align=["left", "right", "left"],
                height=23,
            ),
        )
    ]
)
fig.update_layout(
    title=f"{INDEX_NAME} — Realised Variance And Standard Deviation, {sessions.min():%b %Y} – {sessions.max():%b %Y}",
    height=52 + 23 * len(labels) + 60,
    margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

## VIX Against Realised Volatility, Through Time

One axis, one unit. India VIX and annualised realised σ are both quoted as annualised volatility in percent, which is exactly why they can share a y-axis — this is a comparison of like with like, not two scales forced together.

The realised line is **trailing** `RV_WINDOW`-session σ, so each point is the volatility that had just happened. Read against the VIX line, the picture is the one option sellers rely on: VIX sits above realised most of the time, and the exceptions are clustered, not scattered. When realised punches through VIX it does so in short, violent episodes — which is the shape of the risk being sold.

In [5]:
fig = go.Figure()
series = trailing_rv.dropna()
fig.add_trace(
    go.Scatter(
        x=series.index, y=series, mode="lines", name=f"{INDEX_NAME} realised σ (trailing {RV_WINDOW}d)",
        line=dict(color=INDEX_HEX, width=1.8),
        hovertemplate="%{x|%Y-%m-%d} — %{y:.1f}%<extra>realised</extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=vix.index, y=vix, mode="lines", name="India VIX (implied)",
        line=dict(color=VIX_HEX, width=2),
        hovertemplate="%{x|%Y-%m-%d} — %{y:.1f}%<extra>India VIX</extra>",
    )
)

# Selective direct labels at the right edge, nudged apart so they cannot collide.
ends = sorted([(series.iloc[-1], INDEX_NAME, INDEX_HEX), (vix.iloc[-1], "India VIX", VIX_HEX)])
placed: list[float] = []
for value, label, colour in ends:
    y = value if not placed else max(value, placed[-1] + 1.4)
    placed.append(y)
    fig.add_annotation(
        x=sessions.max(), y=y, text=f" {label} {value:.1f}%", showarrow=False,
        xanchor="left", yanchor="middle", font=dict(color=colour, size=11),
    )

fig.update_layout(
    title=f"India VIX vs {INDEX_NAME} Trailing {RV_WINDOW}-Session Realised σ — Both Annualised, Same Axis",
    template="plotly_white", height=470, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=50, l=60, r=150),
)
fig.update_xaxes(gridcolor=GRID, showline=True, linecolor=GRID)
fig.update_yaxes(title_text="Annualised volatility", ticksuffix="%", gridcolor=GRID, rangemode="tozero")
fig.show()

## The Variance Risk Premium

Now the forward-looking comparison. For every session, India VIX's close is set against the realised σ of the **next** `RV_WINDOW` sessions — the quantity it was forecasting. The premium is reported three ways because option sellers are paid in different units depending on the structure:

- **Vol points** — `VIX − forward RV`. The direct read: how many annualised volatility points the market was charging above what turned up. This is roughly what a straddle seller's edge looks like.
- **Variance points** — `VIX² − forward RV²`, averaged per session. Variance is the additive quantity and what a variance swap settles on, so this is the economically correct premium. It is not a monotone restatement of the vol-point figure: the same 2-point vol spread is worth far more variance at a VIX of 24 than at 11.
- **Variance ratio** — `VIX² / forward RV²`. Unit-free, so it can be compared across regimes. A ratio of 1.0 means the option market was exactly right.

A positive premium means **VIX was rich** — the market charged more than the subsequent realised move justified. The `% sessions VIX rich` column is the hit rate a seller actually experiences, which matters more than the mean: a large average premium earned on 55% of sessions is a different business from a smaller one earned on 80%.

Sessions in the last `RV_WINDOW` of the sample have no full forward window and are excluded.

In [6]:
premium = pd.DataFrame({"vix": vix, "forward": forward_rv}).dropna()
vol_points = premium["vix"] - premium["forward"]
variance_points = premium["vix"] ** 2 - premium["forward"] ** 2
variance_ratio = premium["vix"] ** 2 / premium["forward"] ** 2

summary = pd.DataFrame(
    [
        {
            "Index": INDEX_NAME,
            "Sessions": f"{len(premium):,}",
            "Mean VIX": f"{premium['vix'].mean():.2f}",
            "Forward sigma": f"{premium['forward'].mean():.2f}%",
            "Premium vol": f"{vol_points.mean():+.2f}",
            "Median vol": f"{vol_points.median():+.2f}",
            "Premium variance": f"{variance_points.mean():+,.1f}",
            "Variance ratio": f"{variance_ratio.mean():.2f}×",
            "Pct rich": f"{(vol_points > 0).mean() * 100:.1f}%",
            "Worst shortfall": f"{vol_points.min():+.2f}",
        }
    ]
)

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[80, 70, 50, 105, 90, 90, 130, 75, 95, 130],
            header=dict(
                values=[
                    "Index", "Sessions", "Mean<br>VIX", f"Mean forward<br>{RV_WINDOW}d σ",
                    "Premium<br>(vol pts)", "Median<br>(vol pts)", "Premium<br>(variance pts)",
                    "Variance<br>ratio", "% sessions<br>VIX rich", "Worst shortfall<br>(vol pts)",
                ],
                fill_color=HEADER_FILL, font=dict(color="white", size=11), align="center", height=46,
            ),
            cells=dict(
                values=[summary[c] for c in summary.columns],
                fill_color=[[INDEX_HEX]] + [["white"]] * 9,
                font=dict(color=[["white"]] + [["#333231"]] * 9, size=11),
                align="center", height=28,
            ),
        )
    ]
)
fig.update_layout(
    title=f"India VIX vs {INDEX_NAME} Forward {RV_WINDOW}-Session Realised σ — The Variance Risk Premium",
    height=200, margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

### Was The Premium There Every Month, Or Only On Average?

A mean premium is only tradable if it is persistent. Each bar is one calendar month's mean `VIX − forward RV`, measured against a zero baseline: **warm above zero** means the option market was charging more than the following month delivered, **cool below zero** means realised volatility overran what was priced.

This is the chart that sizes the tail. The months that go cool are the ones where a short-premium book takes its losses, and what matters is how deep they go relative to the many shallow warm months that pay for them.

In [7]:
monthly = vol_points.groupby(premium.index.to_period("M")).mean()
monthly.index = monthly.index.to_timestamp()

fig = go.Figure(
    data=[
        go.Bar(
            x=monthly.index, y=monthly,
            marker=dict(color=[RICH_HEX if v >= 0 else CHEAP_HEX for v in monthly], line=dict(width=0)),
            width=20 * 24 * 3600 * 1000,   # ~20 days, leaving a surface gap between bars
            hovertemplate="%{x|%b %Y} — %{y:+.2f} vol pts<extra></extra>", showlegend=False,
        )
    ]
)
# Identity is carried by position against the baseline and by these labels, not by colour alone.
for text, colour, y in [("VIX rich — premium earned", RICH_HEX, 0.97), ("VIX cheap — realised overran", CHEAP_HEX, 0.03)]:
    fig.add_annotation(
        xref="paper", yref="paper", x=0.005, y=y, text=text, showarrow=False,
        xanchor="left", yanchor="top" if y > 0.5 else "bottom", font=dict(color=colour, size=11),
    )
fig.add_hline(y=0, line_color=MUTED_INK, line_width=1)
fig.update_layout(
    title=f"{INDEX_NAME} — Monthly Mean Variance Risk Premium (VIX − Forward {RV_WINDOW}-Session σ)",
    template="plotly_white", height=420, margin=dict(t=70, b=50, l=60, r=20),
)
fig.update_xaxes(gridcolor=GRID)
fig.update_yaxes(title_text="Vol points", gridcolor=GRID, zeroline=False)
fig.show()

negative = monthly[monthly < 0]
print(f"{len(monthly)} months: {(monthly > 0).sum()} with VIX rich ({(monthly > 0).mean() * 100:.0f}%), {len(negative)} with VIX cheap")
print(f"Mean premium in rich months: {monthly[monthly > 0].mean():+.2f} vol pts | mean shortfall in cheap months: {negative.mean():+.2f} vol pts")
print("Cheap months: " + ", ".join(f"{d:%b %Y} ({v:+.1f})" for d, v in negative.sort_values().items()))

57 months: 46 with VIX rich (81%), 11 with VIX cheap
Mean premium in rich months: +3.63 vol pts | mean shortfall in cheap months: -3.36 vol pts
Cheap months: May 2024 (-6.3), Feb 2026 (-5.8), Feb 2022 (-5.5), Mar 2025 (-5.1), Jan 2022 (-4.1), Mar 2026 (-3.3), Sep 2026 (-2.0), Jan 2026 (-1.8), Apr 2022 (-1.6), Apr 2025 (-0.9), Jul 2024 (-0.6)


## Conditioned On The VIX Regime

The aggregate premium hides the question a seller actually asks: *is the premium better when VIX is low or when it is high?* Low VIX looks cheap in absolute terms but may be fair value; high VIX pays a fat quote but arrives when realised volatility is genuinely dangerous.

Sessions are bucketed by their VIX close into `VIX_BANDS` and each bucket reports what followed. The **VIX-implied one-session σ** column is `VIX / √252` — the daily move the quote implies — set directly beside the mean absolute move the next session actually produced, so the two are comparable without further arithmetic.

The premium column is shaded diverging around zero: **warm = VIX rich**, **cool = VIX cheap**, neutral at fair. Values are printed in every cell, so direction never depends on colour.

In [8]:
def diverging_fill(values: pd.Series) -> list[str]:
    """Blend from the neutral surface toward the rich/cheap pole in proportion to magnitude."""
    peak = values.abs().max() or 1.0
    fills = []
    for value in values:
        pole = RICH_COLOR if value >= 0 else CHEAP_COLOR
        mix = min(abs(value) / peak, 1.0)
        fills.append("rgb({},{},{})".format(*[int(NEUTRAL[i] + (pole[i] - NEUTRAL[i]) * mix) for i in range(3)]))
    return fills


band_labels = ["≤ 12", "12–14", "14–16", "16–18", "18–22", "> 22"]
next_move = returns["close_to_close"].shift(-1).abs() * 100

regime = pd.DataFrame(
    {
        "vix": vix,
        "band": pd.cut(vix, VIX_BANDS, labels=band_labels),
        "implied_daily": vix / ANNUALISE,
        "forward": forward_rv,
        "next_move": next_move,
    }
)

band_rows = []
for label in band_labels:
    bucket = regime[regime["band"] == label]
    with_forward = bucket.dropna(subset=["forward"])
    with_next = bucket.dropna(subset=["next_move"])
    band_rows.append(
        {
            "VIX band": label,
            "Sessions": len(bucket),
            "Share": len(bucket) / len(regime) * 100,
            "Mean VIX": bucket["vix"].mean(),
            "Implied": bucket["implied_daily"].mean(),
            "Actual next": with_next["next_move"].mean(),
            "Forward": with_forward["forward"].mean(),
            "Premium": with_forward["vix"].mean() - with_forward["forward"].mean(),
            "Pct rich": (with_forward["vix"] > with_forward["forward"]).mean() * 100,
        }
    )
bands = pd.DataFrame(band_rows)

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[72, 72, 58, 62, 92, 92, 80, 98, 70],
            header=dict(
                values=[
                    "VIX<br>band", "Sessions", "Share", "Mean<br>VIX", "Implied<br>1-session σ",
                    "Actual mean<br>|next move|", f"Fwd {RV_WINDOW}d<br>σ", "Premium<br>(vol pts)", "%<br>rich",
                ],
                fill_color=HEADER_FILL, font=dict(color="white", size=10), align="center", height=48,
            ),
            cells=dict(
                values=[
                    bands["VIX band"],
                    bands["Sessions"].map("{:,}".format),
                    bands["Share"].map("{:.1f}%".format),
                    bands["Mean VIX"].map("{:.2f}".format),
                    bands["Implied"].map("{:.2f}%".format),
                    bands["Actual next"].map("{:.2f}%".format),
                    bands["Forward"].map("{:.2f}%".format),
                    bands["Premium"].map("{:+.2f}".format),
                    bands["Pct rich"].map("{:.0f}%".format),
                ],
                fill_color=[["white"] * len(bands)] * 7 + [diverging_fill(bands["Premium"]), ["white"] * len(bands)],
                font=dict(size=11, color="#333231"), align="center", height=27,
            ),
        )
    ]
)
fig.update_layout(
    title=f"{INDEX_NAME} — What Followed Each VIX Regime, Implied vs Realised",
    height=52 + 27 * len(bands) + 90, margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

## Is The VIX-Implied One-Session Move Fair?

VIX quotes a 30-day annualised σ. Dividing by √252 converts it to the **one-session σ** the quote implies — the same √t scaling the index construction itself assumes. If that implied σ were an honest description of tomorrow, the share of sessions whose absolute move exceeded `k × σ_implied` would track the Gaussian tail `P(|Z| > k) = erfc(k/√2)`.

Two distinct errors can show up here, and they point in opposite directions:

- **A level error** — implied σ is simply too high or too low, which moves every exceedance rate the same way.
- **A shape error** — the distribution is the wrong shape for its width, which shows up as the gap between measured and benchmark *changing* with `k` instead of staying proportional. The excess kurtosis in the headline table says to expect some of this, and the `Actual ÷ Gaussian` column is where to read it: a constant ratio down the column is a pure level error, a ratio that drifts is a shape error.

The benchmark is drawn in grey as context, not as a second instrument — it is what the market's own pricing convention would predict, and the index line is the measurement.

In [9]:
exceed = pd.DataFrame({"implied": vix / ANNUALISE, "actual": next_move}).dropna()

exceed_rows = []
for multiple in SIGMA_MULTIPLES:
    threshold = multiple * exceed["implied"]
    exceed_rows.append(
        {
            "multiple": multiple,
            "mean_threshold": threshold.mean(),
            "actual": (exceed["actual"] > threshold).mean() * 100,
            "gaussian": math.erfc(multiple / math.sqrt(2)) * 100,
        }
    )
exceedance = pd.DataFrame(exceed_rows)
exceedance["ratio"] = exceedance["actual"] / exceedance["gaussian"]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=exceedance["multiple"], y=exceedance["gaussian"], mode="lines+markers",
        name="Gaussian benchmark", line=dict(color=MUTED_INK, width=1.6, dash="dash"),
        marker=dict(size=7, color=MUTED_INK),
        hovertemplate="%{x}σ — %{y:.2f}% expected<extra>Gaussian</extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=exceedance["multiple"], y=exceedance["actual"], mode="lines+markers", name=INDEX_NAME,
        line=dict(color=INDEX_HEX, width=2),
        marker=dict(size=9, color=INDEX_HEX, line=dict(width=2, color="white")),
        hovertemplate="%{x}σ — %{y:.2f}% of sessions<extra>" + INDEX_NAME + "</extra>",
    )
)
fig.update_layout(
    title=f"{INDEX_NAME} — Share Of Sessions Whose Move Exceeded k × The VIX-Implied One-Session σ",
    template="plotly_white", height=440, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=55, l=60, r=20),
)
fig.update_xaxes(title_text="k (multiples of the VIX-implied one-session σ)", gridcolor=GRID, dtick=0.5)
fig.update_yaxes(title_text="% of sessions exceeded", ticksuffix="%", gridcolor=GRID, rangemode="tozero")
fig.show()

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[62, 112, 105, 105, 118],
            header=dict(
                values=[
                    "k", "Mean threshold<br>(% move)", f"{INDEX_NAME}<br>exceeded",
                    "Gaussian<br>benchmark", "Actual ÷<br>Gaussian",
                ],
                fill_color=HEADER_FILL, font=dict(color="white", size=11), align="center", height=44,
            ),
            cells=dict(
                values=[
                    exceedance["multiple"].map("{:.1f}σ".format),
                    exceedance["mean_threshold"].map("{:.2f}%".format),
                    exceedance["actual"].map("{:.2f}%".format),
                    exceedance["gaussian"].map("{:.2f}%".format),
                    exceedance["ratio"].map("{:.2f}×".format),
                ],
                fill_color=[
                    ["white"] * len(exceedance),
                    ["white"] * len(exceedance),
                    diverging_fill(exceedance["gaussian"] - exceedance["actual"]),
                    ["white"] * len(exceedance),
                    diverging_fill(1 - exceedance["ratio"]),
                ],
                font=dict(size=11, color="#333231"), align="center", height=27,
            ),
        )
    ]
)
fig.update_layout(
    title="Exceedance Table — Warm Where The Market Over-Priced The Move, Cool Where It Under-Priced It",
    height=52 + 27 * len(exceedance) + 90, margin=dict(t=60, b=10, l=10, r=10),
)
fig.show()

mean_ratio = exceed["actual"].mean() / exceed["implied"].mean()
print(f"Mean VIX-implied one-session σ: {exceed['implied'].mean():.3f}%")
print(f"Mean actual |next-session move|: {exceed['actual'].mean():.3f}%")
print(
    f"\nFor a Gaussian, E|r| = σ·√(2/π) = {math.sqrt(2 / math.pi):.3f}·σ. "
    f"Observed ratio E|r|/σ_implied = {mean_ratio:.3f}, "
    f"so the implied σ ran about {(math.sqrt(2 / math.pi) / mean_ratio - 1) * 100:.0f}% richer than the realised move warranted."
)

Mean VIX-implied one-session σ: 0.941%
Mean actual |next-session move|: 0.632%

For a Gaussian, E|r| = σ·√(2/π) = 0.798·σ. Observed ratio E|r|/σ_implied = 0.672, so the implied σ ran about 19% richer than the realised move warranted.


## How Good A Forecast Is VIX, Really?

A persistent premium does not require VIX to be a *good* forecast — only a biased one. These two properties are separate and both matter:

- **Bias** is the premium already measured. The fitted intercept and slope say where it lives: a slope below 1 means VIX over-reacts, pricing more incremental risk than each extra VIX point actually delivers.
- **Skill** is R² — how much of the variation in the next month's realised σ VIX explains at all. A biased forecast with real skill is the ideal thing to sell against, because the bias is the edge and the skill tells you when to size down.

Two reference lines: the grey **y = x** is where VIX would be a perfectly fair forecast — points below it are sessions where VIX was rich — and the dark line is the fit. The final row of the table is the tell for a forecast that is substantially backward-looking: if VIX correlates more tightly with *trailing* realised σ than with the forward σ it is supposed to predict, it is describing the recent past as much as the future.

In [10]:
slope, intercept = np.polyfit(premium["vix"], premium["forward"], 1)
forward_corr = premium["vix"].corr(premium["forward"])
trailing_aligned = pd.concat([vix, trailing_rv], axis=1).dropna()
trailing_corr = trailing_aligned.iloc[:, 0].corr(trailing_aligned.iloc[:, 1])
span = np.array([premium["vix"].min(), premium["vix"].max()])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=span, y=span, mode="lines", name="y = x (VIX exactly fair)",
        line=dict(color=MUTED_INK, width=1.6, dash="dash"), hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=premium["vix"], y=premium["forward"], mode="markers", name=f"{INDEX_NAME} session",
        marker=dict(size=5, color=INDEX_HEX, opacity=0.40),
        customdata=premium.index.strftime("%Y-%m-%d"),
        hovertemplate="%{customdata}<br>VIX %{x:.2f} → forward σ %{y:.2f}%<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=span, y=intercept + slope * span, mode="lines",
        name=f"fit: {intercept:+.2f} + {slope:.3f}·VIX",
        line=dict(color="#0d366b", width=2.5), hoverinfo="skip",
    )
)
fig.update_layout(
    title=f"India VIX vs The Forward {RV_WINDOW}-Session Realised σ It Was Forecasting — {INDEX_NAME}",
    template="plotly_white", height=520, hovermode="closest",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=55, l=60, r=20),
)
fig.update_xaxes(title_text="India VIX close (annualised %)", gridcolor=GRID)
fig.update_yaxes(title_text=f"Realised σ over the next {RV_WINDOW} sessions (annualised %)", gridcolor=GRID)
fig.show()

display(
    pd.DataFrame(
        [
            ("Slope on VIX", f"{slope:.3f}", "below 1 ⇒ VIX over-reacts per point"),
            ("Intercept", f"{intercept:+.2f}", ""),
            (f"corr(VIX, forward {RV_WINDOW}d σ)", f"{forward_corr:.3f}", "forecasting skill"),
            ("R²", f"{forward_corr**2:.3f}", f"{forward_corr**2 * 100:.0f}% of next-month σ explained"),
            (f"corr(VIX, trailing {RV_WINDOW}d σ)", f"{trailing_corr:.3f}", "higher ⇒ partly backward-looking"),
            ("Fair-value VIX at the mean", f"{premium['forward'].mean():.2f}", "where VIX would price no premium"),
        ],
        columns=["Property", "Value", "Note"],
    )
    .style.hide(axis="index")
    .set_caption(f"{INDEX_NAME} — India VIX as a forecast")
)

Property,Value,Note
Slope on VIX,0.864,below 1 ⇒ VIX over-reacts per point
Intercept,-0.29,
"corr(VIX, forward 21d σ)",0.643,forecasting skill
R²,0.413,41% of next-month σ explained
"corr(VIX, trailing 21d σ)",0.750,higher ⇒ partly backward-looking
Fair-value VIX at the mean,12.63,where VIX would price no premium


## VIX's Own Variance

VIX is itself a time series with a variance, and for anyone short options that second-order risk is the one that causes damage. A short strangle loses on the move; a short vega book loses on the *re-pricing*, which arrives faster.

Three properties are measured:

- **Dispersion of the level** — mean, median and standard deviation of VIX itself, which sets what "normal" means and shows how skewed the distribution is toward low readings with a long right tail.
- **Vol-of-vol** — the annualised σ of `ln(VIXₜ / VIXₜ₋₁)`. VIX moves in percentage terms several times faster than the index it describes.
- **The leverage effect** — correlation between the daily change in VIX and the index's own return. A strongly negative value is the mechanism behind short-put pain: the index falls and the volatility you are short reprices upward in the same session.

**Mean reversion** is the fourth property, estimated as the AR(1) coefficient on the level. A coefficient below 1 means shocks decay, and the implied half-life says how long a spike takes to bleed out.

The histogram uses one hue, light to dark by VIX level, with the `VIX_BANDS` edges from the regime table marked so the two sections read against each other.

In [11]:
vix_change = np.log(vix).diff().dropna()
level_change = vix.diff()

counts, edges = np.histogram(vix, bins=np.arange(math.floor(vix.min()), math.ceil(vix.max()) + 1, 1.0))
centres = (edges[:-1] + edges[1:]) / 2

fig = go.Figure(
    data=[
        go.Bar(
            x=centres, y=counts,
            marker=dict(color=centres, colorscale=SEQUENTIAL, showscale=False, line=dict(width=0)),
            width=0.88,   # leaves a surface gap between bars
            hovertemplate="VIX %{x:.0f} — %{y} sessions<extra></extra>", showlegend=False,
        )
    ]
)
for edge in VIX_BANDS[1:-1]:
    fig.add_vline(x=edge, line_color=MUTED_INK, line_width=1)
fig.add_annotation(
    xref="paper", x=0.99, yref="paper", y=0.97,
    text=f"median {vix.median():.1f} · mean {vix.mean():.1f} · σ {vix.std():.1f}",
    showarrow=False, xanchor="right", font=dict(color="#333231", size=11),
)
fig.update_layout(
    title="Distribution Of The India VIX Close — Vertical Rules Mark The Regime Bands",
    template="plotly_white", height=400, bargap=0, margin=dict(t=70, b=55, l=60, r=20),
)
fig.update_xaxes(title_text="India VIX close (annualised %)", gridcolor=GRID)
fig.update_yaxes(title_text="Sessions", gridcolor=GRID, rangemode="tozero")
fig.show()

autocorr = vix.autocorr(lag=1)
half_life = math.log(0.5) / math.log(autocorr) if 0 < autocorr < 1 else float("nan")
leverage = pd.concat([level_change, returns["close_to_close"]], axis=1).dropna()
leverage_corr = leverage.iloc[:, 0].corr(leverage.iloc[:, 1])

display(
    pd.DataFrame(
        [
            ("Mean / median level", f"{vix.mean():.2f} / {vix.median():.2f}"),
            ("Standard deviation of the level", f"{vix.std():.2f} vol pts"),
            ("Variance of the level", f"{vix.var():.2f} vol pts²"),
            ("Min / max", f"{vix.min():.2f} / {vix.max():.2f}"),
            ("Interquartile range", f"{vix.quantile(0.25):.2f} – {vix.quantile(0.75):.2f}"),
            ("Daily σ of log change (vol-of-vol)", f"{vix_change.std() * 100:.2f}%"),
            ("Annualised vol-of-vol", f"{vix_change.std() * ANNUALISE * 100:.1f}%"),
            ("Largest one-session rise / fall", f"{level_change.max():+.2f} / {level_change.min():+.2f} vol pts"),
            ("AR(1) coefficient on the level", f"{autocorr:.4f}"),
            ("Implied shock half-life", f"{half_life:.1f} sessions"),
            (f"corr(ΔVIX, {INDEX_NAME} return) — leverage effect", f"{leverage_corr:+.3f}"),
        ],
        columns=["Property of India VIX", "Value"],
    )
    .style.hide(axis="index")
    .set_caption("India VIX as a series in its own right")
)

Property of India VIX,Value
Mean / median level,14.94 / 13.90
Standard deviation of the level,3.78 vol pts
Variance of the level,14.30 vol pts²
Min / max,9.13 / 32.75
Interquartile range,12.08 – 17.03
Daily σ of log change (vol-of-vol),5.51%
Annualised vol-of-vol,87.5%
Largest one-session rise / fall,+9.09 / -6.84 vol pts
AR(1) coefficient on the level,0.9667
Implied shock half-life,20.4 sessions


## Where Inside The Session The Variance Sits

The final question is practical: the daily numbers say *how much* risk there is, but an option seller choosing an entry or adjustment hour needs to know *when* it arrives.

Mean absolute hourly log return is plotted per candle of the session, split into **VIX terciles**. Terciles are an ordered scale, so the colour job is sequential — one hue, light for the calm third of sessions through dark for the most stressed — rather than three unrelated categorical hues.

Two things to read off it: the shape within a session (how much steeper the first candles are than the middle), and whether a higher VIX regime simply lifts the whole curve or changes that shape. If it only lifts the curve, a single intraday timing rule holds across regimes and only position size needs to respond to VIX — the `High ÷ Low` column in the table is the direct test.

**Read the 15:15 point with care.** That candle covers 15:15–15:30, a quarter of the span of every other slot, so a smaller mean move there is largely mechanical rather than evidence that the close is calm. The six full-hour slots are comparable with each other; the last one is not comparable with them.

In [12]:
TERCILE_LABELS = ["Low VIX third", "Middle third", "High VIX third"]

candles = hourly[INDEX_NAME]
profile = pd.DataFrame(
    {
        "abs_return": hourly_log_returns(INDEX_NAME).abs() * 100,
        "session": candles["session"],
        "slot": candles.index.strftime("%H:%M"),
    }
).dropna()
profile["tercile"] = pd.qcut(profile["session"].map(vix), 3, labels=TERCILE_LABELS)

slots = sorted(profile["slot"].unique())
cuts = vix.quantile([1 / 3, 2 / 3])

fig = go.Figure()
for label, colour in zip(TERCILE_LABELS, TERCILE_HEX):
    means = profile[profile["tercile"] == label].groupby("slot")["abs_return"].mean().reindex(slots)
    fig.add_trace(
        go.Scatter(
            x=slots, y=means, mode="lines+markers", name=label,
            line=dict(color=colour, width=2),
            marker=dict(size=8, color=colour, line=dict(width=2, color="white")),
            hovertemplate="%{x} — %{y:.3f}%<extra>" + label + "</extra>",
        )
    )
fig.update_layout(
    title=(
        f"{INDEX_NAME} — Mean Absolute Hourly Move By Candle, Split By VIX Tercile "
        f"(cuts at {cuts.iloc[0]:.1f} and {cuts.iloc[1]:.1f})"
    ),
    template="plotly_white", height=450, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(t=80, b=55, l=60, r=20),
)
fig.update_xaxes(title_text="Candle start (IST) — first slot is that candle's own open-to-close", gridcolor=GRID)
fig.update_yaxes(title_text="Mean |hourly log return|", ticksuffix="%", gridcolor=GRID, rangemode="tozero")
fig.show()

wide = (
    profile.pivot_table(index="slot", columns="tercile", values="abs_return", aggfunc="mean", observed=True)
    .reindex(slots)
)
wide["High ÷ Low"] = wide["High VIX third"] / wide["Low VIX third"]
display(
    wide.style.format({**{c: "{:.3f}%" for c in TERCILE_LABELS}, "High ÷ Low": "{:.2f}×"})
    .set_caption("Mean |hourly move| per candle by VIX tercile — a flat final column means VIX scales the level, not the shape")
)

tercile,Low VIX third,Middle third,High VIX third,High ÷ Low
slot,,,,
09:15,0.204%,0.285%,0.356%,1.75×
10:15,0.126%,0.172%,0.208%,1.64×
11:15,0.106%,0.147%,0.206%,1.94×
12:15,0.108%,0.147%,0.210%,1.96×
13:15,0.112%,0.151%,0.228%,2.03×
14:15,0.153%,0.180%,0.277%,1.81×
15:15,0.062%,0.072%,0.096%,1.53×


## Takeaways

Computed from the figures above, so this section cannot drift from the data.

In [13]:
one_sigma = exceedance.loc[exceedance["multiple"] == 1.0].iloc[0]
two_sigma = exceedance.loc[exceedance["multiple"] == 2.0].iloc[0]
three_sigma = exceedance.loc[exceedance["multiple"] == 3.0].iloc[0]
annual_sigma = daily_returns.std() * ANNUALISE * 100
gk = garman_klass_sigma(returns)

display(Markdown(f"""
**{INDEX_NAME}**, {len(sessions):,} regular sessions, {sessions.min():%d %b %Y} to {sessions.max():%d %b %Y}.

1. **The level.** Annualised close-to-close σ of **{annual_sigma:.2f}%** — daily σ
   {daily_sigma_pct:.4f}%, daily variance {daily_variance:.4f} %², annualised variance
   {daily_variance * TRADING_DAYS:,.0f} %². The mean absolute daily move was
   {daily_returns.abs().mean() * 100:.3f}% against a median of
   {daily_returns.abs().median() * 100:.3f}%, with skew {daily_returns.skew():+.2f} and excess kurtosis
   {daily_returns.kurtosis():+.2f}: a distribution that is mostly quiet, tilted toward larger down moves,
   and fat-tailed at both ends.

2. **About {overnight.var() / daily_var_raw * 100:.0f}% of daily variance arrives overnight**
   (annualised overnight σ {overnight.std() * ANNUALISE * 100:.2f}% against intraday
   {intraday.std() * ANNUALISE * 100:.2f}%). The overnight and intraday shares sum past 100% because the cross
   term is *negative* ({cross_term / daily_var_raw * 100:+.1f}% of daily variance): the session partly retraces
   its own gap, so the two components offset rather than compound. This is why the range estimators read low —
   Parkinson {parkinson_sigma(returns):.2f}% and Garman–Klass {gk:.2f}% against close-to-close
   {annual_sigma:.2f}% — they measure open-to-close travel and cannot see the gap. Quoting a range-based σ as
   "the" volatility understates the risk a held position carries by roughly
   {(1 - gk / annual_sigma) * 100:.0f}%.

3. **The variance risk premium is large and persistent.** Mean VIX {premium['vix'].mean():.2f} against mean
   forward {RV_WINDOW}-session realised σ {premium['forward'].mean():.2f}% — **{vol_points.mean():+.2f} vol
   points**, or {variance_points.mean():+,.1f} variance points, with VIX rich on
   **{(vol_points > 0).mean() * 100:.1f}%** of sessions and in {(monthly > 0).sum()} of {len(monthly)} calendar
   months. It was positive in *every* VIX band ({bands['Premium'].min():+.2f} to
   {bands['Premium'].max():+.2f} vol points), so the edge is not confined to one regime — but the worst single
   session shortfall was {vol_points.min():+.2f} vol points, which is the number that sizes the book.

4. **VIX over-prices the body of the distribution, but not its extreme tail.** Implied one-session σ averaged
   {exceed['implied'].mean():.3f}% while the actual next-session move averaged {exceed['actual'].mean():.3f}%.
   Moves exceeded 1σ on **{one_sigma['actual']:.1f}%** of sessions against a Gaussian
   {one_sigma['gaussian']:.1f}% — a ratio of {one_sigma['ratio']:.2f}× — and the over-pricing is widest at 2σ,
   where {two_sigma['actual']:.2f}% of sessions breached against {two_sigma['gaussian']:.2f}% expected
   ({two_sigma['ratio']:.2f}×). By 3σ the gap has closed: {three_sigma['actual']:.2f}% versus
   {three_sigma['gaussian']:.2f}% ({three_sigma['ratio']:.2f}×). So this is **not** a pure level error. The
   excess kurtosis of {daily_returns.kurtosis():+.2f} shows up exactly where it should: relative to a Gaussian
   of the implied width, the distribution has a thin shoulder and a tail that catches back up. The premium
   lives in selling the body — the far wing was priced about right.

5. **VIX has real skill, and a slope well below 1.** Fit
   `forward σ = {intercept:+.2f} + {slope:.3f} · VIX`, R² = {forward_corr**2:.3f}. It explains about
   {forward_corr**2 * 100:.0f}% of next-month realised σ — genuinely informative — but each extra VIX point
   buys only {slope:.2f} points of realised volatility, which is the over-reaction that funds short premium.
   Note also that VIX correlates more tightly with *trailing* σ ({trailing_corr:.3f}) than with the forward σ
   it is supposed to predict ({forward_corr:.3f}): it is substantially a backward-looking index.

6. **VIX itself moves far faster than the index.** Annualised vol-of-vol
   {vix_change.std() * ANNUALISE * 100:.0f}% against the index's {annual_sigma:.1f}%, with a leverage-effect
   correlation of {leverage_corr:+.3f} — the index falls and short vega reprices against you in the same
   session. AR(1) of {autocorr:.3f} puts the half-life of a spike at about {half_life:.0f} sessions, so the
   re-pricing is sharp on the way in and then decays slowly: roughly a month for half a shock to bleed out,
   not a few days.

**Caveats.** The sample starts 2022-01-03 — both dumps begin there despite the `2020-01-01` file names, so this
covers {(sessions.max() - sessions.min()).days / 365.25:.1f} years and no pre-2022 stress (not the 2020 crash,
not 2018). VIX is a 30-calendar-day index compared here against {RV_WINDOW} trading sessions and, in the
exceedance test, scaled to one session by √t; both are the index's own conventions, not exact identities.
Realised σ is computed on log closes with no drift adjustment, which is standard and immaterial at a daily mean
return of {daily_returns.mean() * 100:+.4f}%. Index levels carry no dividend adjustment, and the premium
measured here is before the costs — spread, slippage, taxes — that decide whether it is actually harvestable.
"""))


**NIFTY 50**, 1,182 regular sessions, 03 Jan 2022 to 22 Oct 2026.

1. **The level.** Annualised close-to-close σ of **13.64%** — daily σ
   0.8592%, daily variance 0.7383 %², annualised variance
   186 %². The mean absolute daily move was
   0.632% against a median of
   0.489%, with skew -0.40 and excess kurtosis
   +3.85: a distribution that is mostly quiet, tilted toward larger down moves,
   and fat-tailed at both ends.

2. **About 45% of daily variance arrives overnight**
   (annualised overnight σ 9.15% against intraday
   10.87%). The overnight and intraday shares sum past 100% because the cross
   term is *negative* (-8.3% of daily variance): the session partly retraces
   its own gap, so the two components offset rather than compound. This is why the range estimators read low —
   Parkinson 10.68% and Garman–Klass 10.60% against close-to-close
   13.64% — they measure open-to-close travel and cannot see the gap. Quoting a range-based σ as
   "the" volatility understates the risk a held position carries by roughly
   22%.

3. **The variance risk premium is large and persistent.** Mean VIX 14.96 against mean
   forward 21-session realised σ 12.63% — **+2.33 vol
   points**, or +52.6 variance points, with VIX rich on
   **79.6%** of sessions and in 46 of 57 calendar
   months. It was positive in *every* VIX band (+1.67 to
   +4.40 vol points), so the edge is not confined to one regime — but the worst single
   session shortfall was -12.54 vol points, which is the number that sizes the book.

4. **VIX over-prices the body of the distribution, but not its extreme tail.** Implied one-session σ averaged
   0.941% while the actual next-session move averaged 0.632%.
   Moves exceeded 1σ on **21.5%** of sessions against a Gaussian
   31.7% — a ratio of 0.68× — and the over-pricing is widest at 2σ,
   where 2.03% of sessions breached against 4.55% expected
   (0.45×). By 3σ the gap has closed: 0.25% versus
   0.27% (0.94×). So this is **not** a pure level error. The
   excess kurtosis of +3.85 shows up exactly where it should: relative to a Gaussian
   of the implied width, the distribution has a thin shoulder and a tail that catches back up. The premium
   lives in selling the body — the far wing was priced about right.

5. **VIX has real skill, and a slope well below 1.** Fit
   `forward σ = -0.29 + 0.864 · VIX`, R² = 0.413. It explains about
   41% of next-month realised σ — genuinely informative — but each extra VIX point
   buys only 0.86 points of realised volatility, which is the over-reaction that funds short premium.
   Note also that VIX correlates more tightly with *trailing* σ (0.750) than with the forward σ
   it is supposed to predict (0.643): it is substantially a backward-looking index.

6. **VIX itself moves far faster than the index.** Annualised vol-of-vol
   87% against the index's 13.6%, with a leverage-effect
   correlation of -0.668 — the index falls and short vega reprices against you in the same
   session. AR(1) of 0.967 puts the half-life of a spike at about 20 sessions, so the
   re-pricing is sharp on the way in and then decays slowly: roughly a month for half a shock to bleed out,
   not a few days.

**Caveats.** The sample starts 2022-01-03 — both dumps begin there despite the `2020-01-01` file names, so this
covers 4.8 years and no pre-2022 stress (not the 2020 crash,
not 2018). VIX is a 30-calendar-day index compared here against 21 trading sessions and, in the
exceedance test, scaled to one session by √t; both are the index's own conventions, not exact identities.
Realised σ is computed on log closes with no drift adjustment, which is standard and immaterial at a daily mean
return of +0.0213%. Index levels carry no dividend adjustment, and the premium
measured here is before the costs — spread, slippage, taxes — that decide whether it is actually harvestable.
